In [74]:
import yaml
import json
from os import getenv
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime

from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

from draw_bn import draw_bayesian_network
from format_cpts import format_discrete_cpds
from format_queries import format_probability_query

# Load .env file from current directory
load_dotenv()

True

In [57]:
# Create the Bayesian Network structure
bn = DiscreteBayesianNetwork([('V0', 'V1'), ('V0', 'V2'), ('V0', 'V3'), ('V1', 'V3')])

# Define CPDs based on the provided tables

# CPD for V0 (root node)
cpd_v0 = TabularCPD(
    variable='V0',
    variable_card=2,
    values=[[0.5072], [0.4928]],
    state_names={'V0': ['s0', 's1']}
)

# CPD for V1 (depends on V0)
cpd_v1 = TabularCPD(
    variable='V1',
    variable_card=2,
    values=[[0.3110, 0.0704],
            [0.6890, 0.9296]],
    evidence=['V0'],
    evidence_card=[2],
    state_names={'V1': ['s0', 's1'], 'V0': ['s0', 's1']}
)

# CPD for V2 (depends on V0)
cpd_v2 = TabularCPD(
    variable='V2',
    variable_card=2,
    values=[[0.8950, 0.0562],
            [0.1050, 0.9438]],
    evidence=['V0'],
    evidence_card=[2],
    state_names={'V2': ['s0', 's1'], 'V0': ['s0', 's1']}
)

# CPD for V3 (depends on V0 and V1)
cpd_v3 = TabularCPD(
    variable='V3',
    variable_card=2,
    values=[[0.0607, 0.8173, 0.8890, 0.2251],
            [0.9393, 0.1827, 0.1110, 0.7749]],
    evidence=['V0', 'V1'],
    evidence_card=[2, 2],
    state_names={'V3': ['s0', 's1'], 'V0': ['s0', 's1'], 'V1': ['s0', 's1']}
)

# Add CPDs to the bn
bn.add_cpds(cpd_v0, cpd_v1, cpd_v2, cpd_v3)

# Validate the bn
assert bn.check_model()

print("Bayesian Network created successfully!")
print(f"Nodes: {bn.nodes()}")
print(f"Edges: {bn.edges()}")


Bayesian Network created successfully!
Nodes: ['V0', 'V1', 'V2', 'V3']
Edges: [('V0', 'V1'), ('V0', 'V2'), ('V0', 'V3'), ('V1', 'V3')]


In [58]:
# Create inference object
inference = VariableElimination(bn)

# Compute P(V3=s1 | V1=s0)
query_result = inference.query(variables=['V3'], evidence={'V1': 's0'})
prob_v3_s1_given_v1_s0 = query_result.values[1]  # Index 1 corresponds to V3=s1

print(f"\nQuery: P(V3=s1 | V1=s0)")
print(f"Result: {prob_v3_s1_given_v1_s0:.6f}")
print(f"\nFull conditional distribution:")
print(query_result)



Query: P(V3=s1 | V1=s0)
Result: 0.789968

Full conditional distribution:
+--------+-----------+
| V3     |   phi(V3) |
+========+===========+
| V3(s0) |    0.2100 |
+--------+-----------+
| V3(s1) |    0.7900 |
+--------+-----------+


## Let's test LLMs

In [59]:
target = {"V3": "s1"}
evidence = {'V1': 's0'}

query_str = format_probability_query(target, evidence=evidence)
cpts_str = format_discrete_cpds(bn.get_cpds())

# print(query_str)
# print(cpts_str)

#### Prepare LLM call

In [62]:

with open("prompts.yaml", "r") as file:
    prompts = yaml.safe_load(file)

MODEL = "moonshotai/kimi-k2-thinking"

# Initialize OpenAI client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=getenv("OPENROUTER_API_KEY")
)

prompt_str = prompts["prompt_base"].format(cpts=cpts_str, query=query_str)

messages = [
    {"role": "system", "content": prompts["system_prompt"]},
    {"role": "user", "content": prompt_str}
]

print(messages[0]["content"])
print(messages[1]["content"])

You are an assistant that helps with probability reasoning calculations. You will be given some conditional probability tables (CPTs) and a query.

## CPT Format Guide

### 1. Marginal CPTs (nodes with no parents)
```
+----------------+------------+
| Node(Value)    | Probability|
+----------------+------------+
| A(yes)         | 0.3        |
+----------------+------------+
| A(no)          | 0.7        |
+----------------+------------+
```
Each row means **P(Node=Value) = Probability**.

### 2. Conditional CPTs (nodes with one or more parents)
- The **first rows** list all parent nodes
- Each **column** represents one combination of parent values
- **Child rows** give P(child=value | parent assignment) for that column
- Probabilities in each column **sum to 1**

**Example:**
```
+----------+----------+----------+----------+----------+
| A        | A(yes)   | A(yes)   | A(no)    | A(no)    |
+----------+----------+----------+----------+----------+
| B        | B(yes)   | B(no)    | B(

### Run LLM

***Note:** It may take several minutes due to the use of reasoning tokens. For more information about reasoning tokens in OpenRouter: https://openrouter.ai/docs/guides/best-practices/reasoning-tokens

In [63]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    extra_body={
        "reasoning": {
            "effort": "xhigh", # 95% of the tokens are used for reasoning
        }
    },
)

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


In [64]:
reasoning = response.choices[0].message.reasoning
content = response.choices[0].message.content

print(reasoning)
print(content)

 We need to compute P(V3=s1 | V1=s0). The network includes V0, V1, V2, V3. CPTs: V0 is marginal. V1 conditional on V0. V2 conditional on V0. V3 conditional on V0 and V1. No mention of V2 in V3's parents; V3's parents are V0 and V1 only. So V2 is independent of V3 given V0 and V1? Actually V2 is independent of V3 given V0? The network is V0 -> V1, V0 -> V2, V0 and V1 -> V3. So V2 is not a parent of V3, and V2 is independent of V3 given V0 and V1? Since V2 is a child of V0 and not connected to V3, conditional independence holds: given V0 and V1, V2 is independent of V3. So we can ignore V2 for this query.

We need P(V3=s1 | V1=s0). This is a conditional probability where evidence is V1=s0. We need to compute P(V3=s1 and V1=s0) / P(V1=s0). Since V3 depends on V0 and V1, we need to sum over possible values of V0 (s0 or s1). So:

P(V3=s1, V1=s0) = sum_{v0 in {s0,s1}} P(V0=v0) * P(V1=s0 | V0=v0) * P(V3=s1 | V0=v0, V1=s0).

Similarly, P(V1=s0) = sum_{v0} P(V0=v0) * P(V1=s0 | V0=v0). Then rati

In [76]:
def save_response(response):
    
    # Get model name and current timestamp for filename
    model_name = response.model.replace('/', '_')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{model_name}_{timestamp}.json"
    
    # Extract the important data
    data = {
        "model": response.model,
        "provider": response.provider,
        "created": response.created,
        "content": response.choices[0].message.content,
        "reasoning": getattr(response.choices[0].message, 'reasoning', None),
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
            "cost": getattr(response.usage, 'cost', None)
        }
    }
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved to: {filename}")
    return filename

In [78]:
save_response(response)

Saved to: moonshotai_kimi-k2-thinking_20251213_171713.json


'moonshotai_kimi-k2-thinking_20251213_171713.json'